# Define code to initialize APViT

## Step 1: Extract the backbone weights from an APViT checkpoint

In [11]:
import torch

!python ../../weights/convert_weight.py '../../weights/misc/apvit_7class_best.pth' '../../weights/temp/apvit_7class_backbone.pth'
!python ../../weights/convert_weight.py '../../weights/misc/apvit_8class_best.pth' '../../weights/temp/apvit_8class_backbone.pth'

# Check the weights
# Both should have keys that start with 'input_layer.', 'body.0.', 'body.1.', and 'body.2.'
backbone_7class_weights = torch.load('../../weights/temp/apvit_7class_backbone.pth')
backbone_8class_weights = torch.load('../../weights/temp/apvit_8class_backbone.pth')

print('7 class backbone weights:')
for key in backbone_7class_weights.keys():
    print(key)

print('\n\n')
print('8 class backbone weights:')
for key in backbone_8class_weights.keys():
    print(key)

7 class backbone weights:
input_layer.0.weight
input_layer.1.weight
input_layer.1.bias
input_layer.1.running_mean
input_layer.1.running_var
input_layer.1.num_batches_tracked
input_layer.2.weight
body.0.0.res_layer.0.weight
body.0.0.res_layer.0.bias
body.0.0.res_layer.0.running_mean
body.0.0.res_layer.0.running_var
body.0.0.res_layer.0.num_batches_tracked
body.0.0.res_layer.1.weight
body.0.0.res_layer.2.weight
body.0.0.res_layer.3.weight
body.0.0.res_layer.4.weight
body.0.0.res_layer.4.bias
body.0.0.res_layer.4.running_mean
body.0.0.res_layer.4.running_var
body.0.0.res_layer.4.num_batches_tracked
body.0.1.res_layer.0.weight
body.0.1.res_layer.0.bias
body.0.1.res_layer.0.running_mean
body.0.1.res_layer.0.running_var
body.0.1.res_layer.0.num_batches_tracked
body.0.1.res_layer.1.weight
body.0.1.res_layer.2.weight
body.0.1.res_layer.3.weight
body.0.1.res_layer.4.weight
body.0.1.res_layer.4.bias
body.0.1.res_layer.4.running_mean
body.0.1.res_layer.4.running_var
body.0.1.res_layer.4.num_batch

## Step 2: Load base mLDG instance with MS1M weights

We'll load the most complete variant of the network (with EF modules and 8 class output) to avoid randomness in weight initialization as much as possible.

In [12]:
from models.ModifiedLDG import load_base_mLDG

mldg_base = load_base_mLDG(pretrained='../../weights/misc/backbone_ir50_ms1m_epoch120.pth', uses_ef_modules=True, num_classes=8)

The model and loaded state dict do not match exactly

size mismatch for output_layer.3.weight: copying a param with shape torch.Size([512, 25088]) from checkpoint, the shape in current model is torch.Size([1024, 25088]).
size mismatch for output_layer.3.bias: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([1024]).
unexpected key in source state_dict: output_layer.4.weight, output_layer.4.bias, output_layer.4.running_mean, output_layer.4.running_var, output_layer.4.num_batches_tracked

missing keys in source state_dict: local.conv1_1.weight, local.bn1_1.weight, local.bn1_1.bias, local.bn1_1.running_mean, local.bn1_1.running_var, local.conv1_2.weight, local.bn1_2.weight, local.bn1_2.bias, local.bn1_2.running_mean, local.bn1_2.running_var, local.conv2_1.weight, local.bn2_1.weight, local.bn2_1.bias, local.bn2_1.running_mean, local.bn2_1.running_var, local.conv2_2.weight, local.bn2_2.weight, local.bn2_2.bias, local.bn2_2.running_mean, 

## Step 3: Save mLDG weights with just the MS1M knowledge

This model will be used in the experiments without APViT's backbone weights.

In [14]:
torch.save(mldg_base.state_dict(), '../../weights/mldg_no_apvit_pretrain.pth')

## Step 4: Load APViT backbone weights and save (7 classes)

In [15]:
from mmcv.runner import load_state_dict

apvit_weights_seven_class = torch.load('../../weights/temp/apvit_7class_backbone.pth')

if 'state_dict' in apvit_weights_seven_class:
    apvit_weights_seven_class = apvit_weights_seven_class['state_dict']

load_state_dict(mldg_base, apvit_weights_seven_class, strict=False)

torch.save(mldg_base.state_dict(), '../../weights/mldg_apvit_7class_pretrain.pth')

The model and loaded state dict do not match exactly

missing keys in source state_dict: body.3.0.shortcut_layer.0.weight, body.3.0.shortcut_layer.1.weight, body.3.0.shortcut_layer.1.bias, body.3.0.shortcut_layer.1.running_mean, body.3.0.shortcut_layer.1.running_var, body.3.0.res_layer.0.weight, body.3.0.res_layer.0.bias, body.3.0.res_layer.0.running_mean, body.3.0.res_layer.0.running_var, body.3.0.res_layer.1.weight, body.3.0.res_layer.2.weight, body.3.0.res_layer.3.weight, body.3.0.res_layer.4.weight, body.3.0.res_layer.4.bias, body.3.0.res_layer.4.running_mean, body.3.0.res_layer.4.running_var, body.3.1.res_layer.0.weight, body.3.1.res_layer.0.bias, body.3.1.res_layer.0.running_mean, body.3.1.res_layer.0.running_var, body.3.1.res_layer.1.weight, body.3.1.res_layer.2.weight, body.3.1.res_layer.3.weight, body.3.1.res_layer.4.weight, body.3.1.res_layer.4.bias, body.3.1.res_layer.4.running_mean, body.3.1.res_layer.4.running_var, body.3.2.res_layer.0.weight, body.3.2.res_layer.0.bias, bo

## Step 5: Load APViT backbone weights and save (8 classes)

In [16]:
from mmcv.runner import load_state_dict

apvit_weights_eight_class = torch.load('../../weights/temp/apvit_8class_backbone.pth')

if 'state_dict' in apvit_weights_eight_class:
    apvit_weights_eight_class = apvit_weights_eight_class['state_dict']

load_state_dict(mldg_base, apvit_weights_eight_class, strict=False)

torch.save(mldg_base.state_dict(), '../../weights/mldg_apvit_8class_pretrain.pth')

The model and loaded state dict do not match exactly

missing keys in source state_dict: body.3.0.shortcut_layer.0.weight, body.3.0.shortcut_layer.1.weight, body.3.0.shortcut_layer.1.bias, body.3.0.shortcut_layer.1.running_mean, body.3.0.shortcut_layer.1.running_var, body.3.0.res_layer.0.weight, body.3.0.res_layer.0.bias, body.3.0.res_layer.0.running_mean, body.3.0.res_layer.0.running_var, body.3.0.res_layer.1.weight, body.3.0.res_layer.2.weight, body.3.0.res_layer.3.weight, body.3.0.res_layer.4.weight, body.3.0.res_layer.4.bias, body.3.0.res_layer.4.running_mean, body.3.0.res_layer.4.running_var, body.3.1.res_layer.0.weight, body.3.1.res_layer.0.bias, body.3.1.res_layer.0.running_mean, body.3.1.res_layer.0.running_var, body.3.1.res_layer.1.weight, body.3.1.res_layer.2.weight, body.3.1.res_layer.3.weight, body.3.1.res_layer.4.weight, body.3.1.res_layer.4.bias, body.3.1.res_layer.4.running_mean, body.3.1.res_layer.4.running_var, body.3.2.res_layer.0.weight, body.3.2.res_layer.0.bias, bo